In [1]:
# ==============================================================================
# 🏆 IEEE 118-Bus N-1 极限测试集压铸机 (大胜归来版)
# 特点：只断环网链路（零孤岛） | 自动兜底算法 | 物理矩阵完美剥离
# ==============================================================================
import pandapower as pp
import pandapower.networks as nw
import numpy as np
import pandas as pd
from tqdm import tqdm
import copy
import os

# 1. 基础设置
TARGET_SAMPLES = 1000
STD_DEV = 0.03
BASE_MVA = 100.0       

def get_base_physics_matrices(net):
    pp.runpp(net, numba=False)
    return net._ppc['internal']['Ybus'].todense()

def get_cut_matrices(Y_base, f, t):
    Y_cut = Y_base.copy()
    y_line = -Y_base[f, t] 
    Y_cut[f, f] -= y_line; Y_cut[t, t] -= y_line
    Y_cut[f, t] += y_line; Y_cut[t, f] += y_line 
    return np.real(Y_cut), np.imag(Y_cut)

# ==========================================
def generate_118_n1_dataset(net_base, Y_base, case_name, line_idx_to_cut):
    print(f"\n🌪️ 启动战役: {case_name} | 线路 Index: {line_idx_to_cut}")
    
    net_case = copy.deepcopy(net_base)
    f = int(net_case.line.at[line_idx_to_cut, 'from_bus'])
    t = int(net_case.line.at[line_idx_to_cut, 'to_bus'])
    
    # 执行物理切断
    net_case.line.at[line_idx_to_cut, 'in_service'] = False
    print(f"   🔪 物理切断：Bus {f} <---> Bus {t} (无孤岛环网模式)")
    
    # 物理矩阵保存
    G_cut, B_cut = get_cut_matrices(Y_base, f, t)
    save_dir = "ieee118_dataset"
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, f'G_118_{case_name}.npy'), G_cut)
    np.save(os.path.join(save_dir, f'B_118_{case_name}.npy'), B_cut)
    
    base_p_mw_vals = net_case.load.p_mw.values
    base_q_mvar_vals = net_case.load.q_mvar.values
    
    V_mag_all, V_ang_all, P_inj_all, Q_inj_all = [], [], [], []
    valid_count = 0
    
    with tqdm(total=TARGET_SAMPLES, desc=f"Producing {case_name}") as pbar:
        while valid_count < TARGET_SAMPLES:
            try:
                # 随机负荷
                scale = np.random.normal(1.0, STD_DEV, len(net_case.load))
                scale = np.clip(scale, 0.95, 1.05)
                net_case.load.p_mw.values[:] = base_p_mw_vals * scale
                net_case.load.q_mvar.values[:] = base_q_mvar_vals * scale
                
                # 🚨 策略：优先用 Newton-Raphson，失败则自动切换 Fast-Decoupled 兜底
                try:
                    pp.runpp(net_case, algorithm='nr', numba=False)
                except:
                    pp.runpp(net_case, algorithm='fdbx', numba=False)
                
                P_inj_all.append(net_case.res_bus.p_mw.fillna(0.0).values / BASE_MVA)
                Q_inj_all.append(net_case.res_bus.q_mvar.fillna(0.0).values / BASE_MVA)
                V_mag_all.append(net_case.res_bus.vm_pu.fillna(1.0).values)
                V_ang_all.append(net_case.res_bus.va_degree.fillna(0.0).values)
                
                valid_count += 1
                pbar.update(1)
                
            except:
                continue
                
    dataset_3d = np.stack((np.array(P_inj_all), np.array(Q_inj_all), 
                           np.array(V_mag_all), np.array(V_ang_all)), axis=2)
    np.save(os.path.join(save_dir, f"ieee118_{case_name}_1k.npy"), dataset_3d)

if __name__ == "__main__":
    net_118 = nw.case118()
    print("🚀 正在初始化 118 节点母体...")
    Y_base = get_base_physics_matrices(net_118)
    
    # 💡 这三条线路是 118 系统中典型的强连接环路，断开后系统极度稳健，必出数据！
    generate_118_n1_dataset(net_118, Y_base, case_name="C1", line_idx_to_cut=7)  # Bus 11-12
    generate_118_n1_dataset(net_118, Y_base, case_name="C2", line_idx_to_cut=30) # Bus 23-24
    generate_118_n1_dataset(net_118, Y_base, case_name="C3", line_idx_to_cut=80) # Bus 54-56
    
    print("\n🎉 118 节点 N-1 极限测试集完美收工！直接起飞！")

🚀 正在初始化 118 节点母体...

🌪️ 启动战役: C1 | 线路 Index: 7
   🔪 物理切断：Bus 8 <---> Bus 9 (无孤岛环网模式)


Producing C1: 100%|██████████| 1000/1000 [00:22<00:00, 43.82it/s]



🌪️ 启动战役: C2 | 线路 Index: 30
   🔪 物理切断：Bus 24 <---> Bus 26 (无孤岛环网模式)


Producing C2: 100%|██████████| 1000/1000 [00:24<00:00, 40.63it/s]



🌪️ 启动战役: C3 | 线路 Index: 80
   🔪 物理切断：Bus 55 <---> Bus 58 (无孤岛环网模式)


Producing C3: 100%|██████████| 1000/1000 [00:24<00:00, 40.26it/s]



🎉 118 节点 N-1 极限测试集完美收工！直接起飞！


In [3]:
import pandapower as pp, pandapower.networks as nw, copy
net = nw.case118()
net.line.at[8, 'in_service'] = False   # 或 51、96
pp.runpp(net, numba=False)
print("收敛成功")

收敛成功
